# Enzyme Function Prediction with CLEAN

## Overview

This notebook is a simplified example of **CLEAN (https://www.science.org/doi/10.1126/science.adf2465)**, a popular method for predicting enzyme function from protein sequences using **contrastive learning**.

---

## What is Enzyme Function Prediction?

**Enzymes** are biological catalysts that accelerate chemical reactions essential for life. Understanding what reactions an enzyme catalyzes is fundamental to:

- **Biotechnology:** Engineering enzymes for industrial applications
- **Drug discovery:** Identifying therapeutic targets and off-target effects
- **Metabolic engineering:** Designing biosynthetic pathways
- **Metagenomics:** Annotating novel enzymes from environmental samples
- **Synthetic biology:** Building new biological systems

---

## Why is This Problem Hard?

Despite decades of research, enzyme function prediction remains challenging:

### 1. The Sequence-Function Gap
- Sequence similarity doesn't always imply functional similarity
- Small changes in sequence can drastically alter substrate specificity
- Convergent evolution creates functionally similar enzymes with low sequence identity

### 2. Experimental Bottleneck
- Experimental characterization is slow and expensive
- Millions of sequences are unannotated
- **>99% of proteins in UniProtKB lack experimental validation**

### 3. Label Scarcity and Noise
- EC (Enzyme Commission) numbers are often incomplete or outdated
- Many enzymes catalyze multiple reactions (promiscuous enzymes)
- Annotation quality varies widely across databases

### 4. Complexity of Chemical Space
- Tens of thousands of known enzymatic reactions
- Each reaction involves specific substrates, cofactors, and conditions
- Traditional methods struggle to capture reaction-level details

---

## What is CLEAN?

**CLEAN** addresses these challenges by learning enzyme representations through **contrastive learning** on biochemical reactions.

### Key Ideas

#### Reaction-Centric Learning
Instead of learning from EC numbers alone, CLEAN learns from the **chemical reactions** enzymes catalyze. This provides:
Let's begin by setting up our environment and loading the data.

## Dataset: CARE (Classification And Retrieval of Enzymes)

**CARE** is a standardized benchmark suite designed to evaluate machine learning methods for enzyme function prediction. It addresses the lack of consistent evaluation protocols in the field by providing curated datasets and rigorous train-test splits.

### Two Core Tasks

**1. EC Classification**  
Predict the Enzyme Commission (EC) number of a protein from its amino acid sequence.

**2. Reaction-to-Enzyme Retrieval**  
Given a chemical reaction, retrieve the EC number(s) of enzymes that catalyze it.

### Key Features

- **Standardized evaluation splits** that test biologically relevant forms of generalization
- **Out-of-distribution scenarios** including novel enzyme families, sequence diversity, and unseen reaction types
- **Baseline implementations** for state-of-the-art methods
- **Open-source and reproducible** experimental protocols

### Why CARE Matters

Before CARE, enzyme function prediction methods were evaluated on inconsistent datasets with varying quality and split strategies, making fair comparison difficult. CARE provides:

- A unified framework for benchmarking
- Realistic evaluation of model generalization
- A foundation for developing new methods

### Reference

CARE introduces **CREEP (Contrastive Reaction-EnzymE Pretraining)** as a baseline for the retrieval task and compares it against methods like CLIPZyme.

**Dataset available at:** https://github.com/jsunn-y/CARE/

---

This notebook uses the CARE dataset to demonstrate enzyme function prediction with contrastive learning approaches.


In [1]:
!curl -X GET \
     "https://datasets-server.huggingface.co/splits?dataset=arianemora%2FAMLD_workshop_ML4Enzymes"

{"splits":[{"dataset":"arianemora/AMLD_workshop_ML4Enzymes","config":"default","split":"train"}],"pending":[],"failed":[]}

In [2]:
# Copy the datasets across from hugging face
! pip install -U huggingface_hub

/bin/bash: line 1: pip: command not found


In [ ]:
! wget -c https://huggingface.co/datasets/arianemora/AMLD_workshop_ML4Enzymes/resolve/main/data_AMLD_workshop_2026.zip
! unzip data_AMLD_workshop_2026.zip

--2026-03-11 14:46:22--  https://huggingface.co/datasets/arianemora/AMLD_workshop_ML4Enzymes/resolve/main/data_AMLD_workshop_2026.zip
Resolving huggingface.co (huggingface.co)... 13.35.202.34, 13.35.202.97, 13.35.202.40, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.34|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/6989b2cae143f25e9292702d/94e4e8c79a2f88d9cad0ee85d01964304db81a98d078b69b36ff5cb56cd73ab0?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27data_AMLD_workshop_2026.zip%3B+filename%3D%22data_AMLD_workshop_2026.zip%22%3B&response-content-type=application%2Fzip&Expires=1773243982&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiRXBvY2hUaW1lIjoxNzczMjQzOTgyfX0sIlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjk4OWIyY2FlMTQzZjI1ZTkyOTI3MDJkLzk0ZTRlOGM3OWEyZjg4ZDljYWQwZWU4NWQwMTk2NDMwNGRiODFhOThkMDc4YjY5YjM2ZmY1Y2I1NmNkNzNhYjBcXD9yZXNwb25zZS1jb250Z

## Running PyTorch in a Google Colab Notebook

Google Colab provides a preconfigured environment with **PyTorch**, **CUDA**, and **GPUs** available at no cost. Follow the steps below to run PyTorch correctly and efficiently.

---

## Enable a GPU Runtime (Recommended)

In the Colab menu:
Runtime → Change runtime type → Hardware accelerator → GPU → Save



In [ ]:
# Check GPU is available
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: Tesla T4


In [ ]:
DATA_DIR = "/content/data_AMLD_workshop_2026/"

In [ ]:
! ls /content/data_AMLD_workshop_2026/

30-50_protein_test.csv
30_protein_test.csv
price_protein_test.csv
protein_train.csv
uniprotkb_reviewed_true_2025_17_02_ESM_3B_embeddings_smaller.pkl


# Install required libraries

In [ ]:
!pip install pandas
# Install torch with cuda
!pip install torch --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


## UniProt: A Core Resource for Enzyme‑Focused Machine Learning

**UniProt (Universal Protein Resource)** is the main knowledgebase for protein sequences and functional annotations, and a useful resource for **enzyme modeling, representation learning, and function prediction**.

For ML researchers, UniProt provides **high‑quality labels, rich metadata, and cross‑database links** needed to train, evaluate, and interpret models on biological sequence data.

**UniProt Home:** https://www.uniprot.org

---

## What UniProt Contains (ML‑Relevant View)

Each UniProt protein entry integrates:

- **Amino acid sequence** (primary input for ML models)
- **Enzyme function and EC numbers**
- **Catalytic activity (reaction equations)**
- **Protein names and synonyms**
- **Taxonomy (organism, lineage)**
- **Domains, motifs, and active sites**
- **Cofactors and metal binding**
- **Cross‑references** (PDB, KEGG, BRENDA, AlphaFold)

These annotations enable **supervised learning**, **multi‑task learning**, and **benchmarking** across enzyme‑related tasks.

---

## UniProt Knowledgebase (UniProtKB)

UniProtKB has two complementary sections:

### UniProtKB/Swiss‑Prot (Reviewed)
- **Manually curated**
- Experimentally supported annotations
- Low redundancy, high label quality

Best choice for:
- Gold‑standard training sets
- Model validation
- Function benchmarking

https://www.uniprot.org/help/swiss-prot

---

### UniProtKB/TrEMBL (Unreviewed)
- Automatically annotated
- Large‑scale coverage
- Some label noise

Best choice for:
- Pretraining
- Representation learning
- Semi‑supervised learning

https://www.uniprot.org/help/trembl

---

## UniProt and Enzymes

UniProt is one of the **primary sources of enzyme annotations**, including:

### EC Numbers
- Hierarchical enzyme classification (EC 1–7)
- Often partial or multiple ECs per protein

https://www.uniprot.org/help/ec_numbers


---

### Catalytic Activity
- Reaction equations written in a controlled vocabulary
- Links to **Rhea**, a curated reaction database

 https://www.rhea-db.org

This enables:
- Reaction‑level prediction tasks
- Mapping sequences → chemistry

---

## Why UniProt Matters for Machine Learning

| ML Task | UniProt Contribution |
|---|---|
| Function prediction | EC numbers, descriptions |
| Representation learning | Millions of sequences |
| Multi‑label classification | Proteins with multiple ECs |
| Transfer learning | Reviewed → unreviewed |
| Model interpretability | Active sites, domains |

---

## Key Cross‑References (Critical for ML Pipelines)

UniProt entries link to:

- **PDB** (3D structures): https://www.rcsb.org
- **AlphaFold DB** (predicted structures): https://alphafold.ebi.ac.uk
- **KEGG** (pathways): https://www.kegg.jp
- **BRENDA** (enzyme kinetics): https://www.brenda-enzymes.org
- **InterPro** (domains): https://www.ebi.ac.uk/interpro/

These links enable **multimodal learning** (sequence + structure + chemistry).

---

## Accessing UniProt Programmatically

### Web Interface
- Advanced query syntax
- Field‑specific filtering (EC, organism, reviewed)

🔗 https://www.uniprot.org/uniprotkb

---

### REST API (Recommended for ML)

```bash
https://rest.uniprot.org/uniprotkb/search?query=ec:1.1.1.1&format=json

In [ ]:
import pandas as pd
import torch
#sub out the df with mine
# read in training data
train_df = pd.read_csv(f'{DATA_DIR}/protein_train.csv')
test_df = pd.read_csv(f'{DATA_DIR}/30_protein_test.csv')

# df with ESM2 embeddings
df = pd.read_pickle(f'{DATA_DIR}/uniprotkb_reviewed_true_2025_17_02_ESM_3B_embeddings_smaller.pkl')

In [ ]:
# Try contrastive model normal mean embeddings
# change it later to pair token embedding
# generate or implement a column activity
# df['activit']
import numpy as np
import random

df['activity'] = [random.random() for i in range(0, len(df))]
df.head(2)

,Entry,embedding,activity
0,A0A023I7E1,"[-0.0038913805, 0.054855403, -0.052346736, 0.0...",0.478218
1,A0A024RXP8,"[-0.061283756, 0.03423623, -0.034157578, -0.10...",0.197915


In [ ]:
train_df

,Unnamed: 0,Entry,Entry Name,Sequence,EC number,Length,EC All,clusterRes30,clusterRes50,clusterRes70,clusterRes90,EC3,EC2,EC1
0,0,A0A009IHW8,ABTIR_ACIB9,MSLEQKKGADIISKILQIQNSIGKTTSPSTLKTKLSEISRKEQENA...,3.2.2.6,269,3.2.2.6,A1AY86,A0A009IHW8,A0A009IHW8,A0A009IHW8,3.2.2,3.2,3
1,1,A0A023I7E1,ENG1_RHIMI,MRFQVIVAAATITMITSYIPGVASQSTSDGDDLFVPVSNFDPKSIF...,3.2.1.39,796,3.2.1.39,D4AZ24,A0A023I7E1,A0A023I7E1,A0A023I7E1,3.2.1,3.2,3
2,2,A0A024SC78,CUTI1_HYPJR,MRSLAILTTLLAGHAFAYPKPAPQSVNRRDWPSINEFLSELAKVMP...,3.1.1.74,248,3.1.1.74,A8QPD8,A8QPD8,A8QPD8,A0A024SC78,3.1.1,3.1,3
3,3,A0A024SH76,GUX2_HYPJR,MIVGILTTLATLATLAASVPLEERQACSSVWGQCGGQNWSGPTCCA...,3.2.1.91,471,3.2.1.91,B2AE04,A1CCN4,A0A024SH76,A0A024SH76,3.2.1,3.2,3
4,4,A0A044RE18,BLI_ONCVO,MYWQLVRILVLFDCLQKILAIEHDSICIADVDDACPEPSHTVMRLR...,3.4.21.75,693,3.4.21.75,Q9VBC7,A0A044RE18,A0A044RE18,A0A044RE18,3.4.21,3.4,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184524,185990,Q05115,AMDA_BORBO,MQQASTPTIGMIVPPAAGLVPADGARLYPDLPFIASGLGLGSVTPE...,4.1.1.76,240,4.1.1.76,Q05115,Q05115,Q05115,Q05115,4.1.1,4.1,4
184525,185991,Q6HX62,Y3032_BACAN,MGQNQFRWSNEQLREHVEIIDGTRSPHKLLKNATYLNSYIREWMQA...,3.5.4.2,584,3.5.4.2,Q9KF49,Q6HX62,Q6HX62,Q6HX62,3.5.4,3.5,3
184526,185992,Q6L032,Y1085_PICTO,MLLKNIKISNDYNIFMIIASRKPSLKDIYKIIKVSKFDEPADLIIE...,3.5.4.2,573,3.5.4.2,Q9KF49,Q6L032,Q6L032,Q6L032,3.5.4,3.5,3
184527,185993,Q94MV8,VG56_BPLZ5,MAHFNECAHLIEGVDKANRAYAENIMHNIDPLQVMLDMQRHLQIRL...,3.6.1.12,172,3.6.1.12,P39262,P39262,Q94MV8,Q94MV8,3.6.1,3.6,3


In [ ]:
test_df

,Entry,Entry Name,Sequence,EC number,Length,EC All,clusterRes30,clusterRes50,clusterRes70,clusterRes90,EC3,EC2,EC1
0,O32178,FADN_BACSU,MHKHIRKAAVLGSGVMGSGIAAHLANIGIPVLLLDIVPNDLTKEEE...,1.1.1.35,789,1.1.1.35,O32178,O32178,O32178,O32178,1.1.1,1.1,1
1,P40580,BZRD_YEAST,MGKVILITGASRGIGLQLVKTVIEEDDECIVYGVARTEAGLQSLQR...,1.1.1.320,263,1.1.1.320,P40580,P40580,P40580,P40580,1.1.1,1.1,1
2,P05707,SRLD_ECOLI,MNQVAVVIGGGQTLGAFLCHGLAAEGYRVAVVDIQSDKAANVAQEI...,1.1.1.140,259,1.1.1.140,P05707,P05707,P05707,P05707,1.1.1,1.1,1
3,P29898,DHM2_PARDE,MKRILTLTVAALALGTPALAYDGTNCKAPGNCWEPKPDYPAKVEGS...,1.1.2.7,103,1.1.2.7,P29898,P29898,P29898,P29898,1.1.2,1.1,1
4,A0A075HNX4,LCAO_UNCAR,MAQGAQRKNFGHNQILRPSAAYTPVDEQEVLQILDRHRGQRIRAVG...,1.1.3.20,479,1.1.3.20,A0A075HNX4,A0A075HNX4,A0A075HNX4,A0A075HNX4,1.1.3,1.1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
427,Q5WCL2,TAGH_SHOC1,MNPKMILTGVSKKYTLYRNNTEKLKAMFFPKTREQHRDFYALKDIN...,7.5.2.4,360,7.5.2.4,Q5WCL2,Q5WCL2,Q5WCL2,Q5WCL2,7.5.2,7.5,7
428,Q3JSI8,RBSA1_BURP1,MRASLENGDDHDAHRLVDAGFRPPGRPRAARRRAFARARRGERRAR...,7.5.2.7,859,7.5.2.7,Q3JSI8,Q3JSI8,Q3JSI8,Q3JSI8,7.5.2,7.5,7
429,O21280,CCMA_RECAM,MNLTKIQNLTIHNITGIRSNKIIFQNINFSLEKGSLFIIQGSNGSG...,7.6.2.5,222,7.6.2.5,O21280,O21280,O21280,O21280,7.6.2,7.6,7
430,B0R5G4,BTUDA_HALS3,MTLDVTGLDVELAGTRILDDVHASIRDGHLVGVVGPNGAGKSTLLR...,7.6.2.8,398,7.6.2.8,B0R5G4,B0R5G4,B0R5G4,B0R5G4,7.6.2,7.6,7


## Setup indicies for CL model

In contrastive learning, the objective is to **learn representations** by pulling **related samples (positives)** together in embedding space and pushing **unrelated samples (negatives)** apart.

In [ ]:
from collections import defaultdict

# Also do the same for proteins
idx_to_embedding = {}
value_to_index = {}
idx_to_label = {}
seq_to_embedding = dict(zip(df['Entry'].values, df['embedding'].values))

def build_idxs(ec_level):
    i = 0
    idx_to_embedding = {}
    value_to_index = {}
    idx_to_label = {} # idx to activity the rest can stay the same
    idx_to_entry = {}
    class_to_indicies = defaultdict(list)

    for entry, ec in train_df[['Entry', 'EC number']].values:# activity instead of EC number
        if seq_to_embedding.get(entry) is not None:
            clipped_embedding = seq_to_embedding.get(entry).flatten()
            idx_to_embedding[i] = clipped_embedding
            idx_to_entry[i] = entry
            value_to_index[entry] = i
            class_number = str('.'.join(ec.split('.')[:ec_level]))
            idx_to_label[i] = class_number
            class_to_indicies[class_number].append(i) # which indices to which class can use it to match correctly -> weighted towards negative data -> sample more from active site
            i += 1
    return idx_to_embedding, idx_to_label, value_to_index, class_to_indicies# get of other things rid and only change activity

# Simplest way of building the test and the training dataset
# build paires
def build_train_test_df(idx_to_embedding, idx_to_label, num_pairs):
    all_pair_embeddings, all_labels = [], []
    num_idxs = len(idx_to_embedding) - 1
    for i in tqdm(range(0, num_pairs)):
        sample_i = random.sample(range(0, num_idxs), 2)
        all_pair_embeddings.append([sample_i[0], sample_i[1]])
        if idx_to_label.get(sample_i[0]) == idx_to_label.get(sample_i[1]):
            all_labels.append(1) # what makes good positive or negative .2 with a diffrence
        else:
            all_labels.append(-1)
    return all_pair_embeddings, all_labels

In [ ]:
#Make pairs of data -> would run out of memory
#def -> data set class -> is an object make actions on it
#class has a few important thing class is an object with rules and functions self is an attribute
#self.columns = colos or def iloc (index) rerurn self values(index)
#data set class Dataset(data) -> self.idx = indicies -> self.data = data
# def get_entry(i)
#return()index=indicies[i] -> return selfdata(i), self.data[2],I]
#

## Enzyme Commission (EC) Numbers

**EC numbers** are a standardized numerical classification system for **enzymes**.

They describe the **chemical reaction an enzyme catalyzes**, not the specific protein sequence.

---

### EC Number Format

An EC number has the form:

Each level provides more specific information about the enzyme’s function.

| Level | Name | Description |
|---|---|---|
| **a** | Class | Broad type of reaction |
| **b** | Subclass | Group or bond acted upon |
| **c** | Sub‑subclass | Specific reaction type |
| **d** | Serial number | Unique enzyme identifier |

---

### Example

**EC 1.1.1.1** — *Alcohol dehydrogenase*

| Level | Value | Meaning |
|---|---|---|
| 1 | Oxidoreductase | Oxidation–reduction reactions |
| 1 | Acts on CH‑OH group | Alcohol group donors |
| 1 | Uses NAD⁺/NADP⁺ | Electron acceptor |
| 1 | Alcohol dehydrogenase | Specific enzyme |

---

### Major EC Classes

| Class | Name | Reaction Type |
|---|---|---|
| **EC 1** | Oxidoreductases | Oxidation–reduction |
| **EC 2** | Transferases | Transfer of functional groups |
| **EC 3** | Hydrolases | Hydrolysis reactions |
| **EC 4** | Lyases | Bond breaking without ATP |
| **EC 5** | Isomerases | Isomerization |
| **EC 6** | Ligases (Synthetases) | Bond formation using ATP |
| **EC 7** | Translocases | Transport across membranes |

---

### Notes

- EC numbers describe **enzyme function**, not gene or protein structure.
- Different proteins can share the **same EC number**.
- Some enzymes have **multiple EC numbers** if they catalyze multiple reactions.
- Incomplete classifications may appear as:


In [ ]:
# Build indicies for level 4 EC # build diffrent activity matches
# idx_to_embedding, idx_to_label, value_to_index, class_to_indicies = build_idxs(4)# dose not matter

In [ ]:
# Imports
import os
import pandas as pd
import torch
from collections import defaultdict
from tqdm import tqdm
from torch.utils.data import Dataset
import torch.nn as nn
import torch.nn.functional as F
# Setup the dataset
import numpy as np
from torch.utils.data import DataLoader

# Simple Encoder Network
class SimpleNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.01):# one dimensional  n = -> get output embeddings 128
        super(SimpleNetwork, self).__init__()
        # self.protein_layer = nn.Linear(input_dim, hidden_dim)
        # self.reaction_layer = nn.Linear(input_dim, hidden_dim)
        self.hidden_layer = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(hidden_dim)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, input_type='protein'):# -> Any:
        if input_type == 'protein':
            x = self.protein_layer(x)
        else:
            x = self.reaction_layer(x)
        x = self.norm(x)
        x = self.relu(x)
        x = self.hidden_layer(x)
        return x
#x is the mean squared embedding for me
# Contrastive Loss Function
class ContrastiveLoss(nn.Module):

    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, z1, z2, labels, margin=1.0, pos_weight=1.0, neg_weight=1.0):
        """
        z1, z2: (N, D) - two batches of embeddings
        labels: (N,) - 1 for similar, -1 for dissimilar
        margin: minimal required distance for dissimilar pairs
        """
        # Compute Euclidean distances
        distances = F.pairwise_distance(z1, z2, p=2)

        # Similar pairs (label == 1): minimize squared distance
        pos_loss = pos_weight*(distances ** 2)

        # Dissimilar pairs (label == -1): maximize distance up to margin
        neg_loss = neg_weight * F.relu(margin - distances) ** 2

        # Combine losses
        loss = torch.where(labels == 1, pos_loss, neg_loss)
        return loss.mean()

# Setup for contastive loss
class PointerPairedDataset(Dataset):
    def __init__(self, main_store, pair_indices, labels):
        self.main_store = main_store     # Dict - e.g.  {i: torch.rand(512) for i in range(1000)}
        self.pair_indices = pair_indices # Each pair is a tuple of keys from `main_store`, like (0, 5) or (2, 8)
        self.labels = labels             # List of labels for each pair

    def __len__(self):
        return len(self.pair_indices)

    def __getitem__(self, idx):
        # Retrieve indices for the current pair
        idx1, idx2 = self.pair_indices[idx]
        # Look up the actual data in the main store using these indices
        item1, item2 = self.main_store[idx1], self.main_store[idx2]
        label = self.labels[idx]
        return item1, item2, label  # Return the pair for contrastive training


# Define the dataset class
class PairedEmbeddingsDataset(Dataset):
    def __init__(self, tensor1, tensor2, labels):
        #assert tensor1.shape == tensor2.shape, "The two tensors must have the same shape"
        assert tensor1.shape[0] == len(labels), "The number of labels must match the number of rows in the tensors"

        self.tensor1 = tensor1
        self.tensor2 = tensor2
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.tensor(self.tensor1[idx], dtype=torch.float32), \
               torch.tensor(self.tensor2[idx], dtype=torch.float32), \
               self.labels[idx]


class BasicDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.labels[idx]
        return x, y

def compute_all_train_embeddings(model, train_dataset):
    """Returns all training embeddings and their labels"""
    embeddings = []
    labels = []
    model.eval()

    with torch.no_grad():
        for x, y in tqdm(train_dataset):
            emb = model.forward(x.unsqueeze(0))  # Add batch dimension
            embeddings.append(emb)
            labels.append(y)

    return embeddings, labels

def classify_with_softmax(train_embs, train_labels, test_emb, num_classes):
    distances = torch.norm(train_embs - test_emb, dim=1)
    similarities = -distances  # negative for similarity
    probs = F.softmax(similarities, dim=0)

    # Aggregate class-wise probabilities
    class_probs = torch.zeros(num_classes)
    for prob, label in zip(probs, train_labels):
        class_probs[label] += prob

    return class_probs / class_probs.sum()

def classify_by_nearest(train_embs, train_labels, test_emb, top_k=1):
    # Compute distances to all training embeddings
    distances = torch.norm(train_embs - test_emb, dim=1)  # Euclidean

    # Get top-k closest
    topk_indices = torch.topk(-distances, k=top_k).indices  # negative for closest

    # Get the labels
    topk_labels = train_labels[topk_indices]
    print(topk_indices[:2])
    # For top-1 NN classification
    if top_k == 1:
        print(topk_labels[0])
        return topk_labels[0].item()

    # For majority voting
    predicted_label = torch.mode(topk_labels).values.item()
    return predicted_label

def evaluate_nearest_neighbor(model, train_dataset, test_dataset):
    train_embs, train_labels = compute_all_train_embeddings(model, train_dataset)
    print(train_labels[0])
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y_true in test_dataset:
            test_emb = model.forward(x.unsqueeze(0)).squeeze(0)
            y_pred = classify_by_nearest(train_embs, train_labels, test_emb)
            print(y_pred, y_true)
            correct += (y_pred == y_true)
            total += 1

    acc = correct / total
    print(acc)
    return acc

# train the model for some epochs

In [ ]:
import random

num_pairs = 1000000 # You can see that increasing the number of pairs will improve the performance
batch_size = 1000 # We keep this small for the colab notebook but usually this would be larger
input_dim = 2560 # This is specific to the ESM2 embedding that we're using
hidden_dim = 1024 # you can change these as you like
hidden_dim_2 = 256
latent_dim = 512
lr = 0.001
epochs = 5

num_models = 1
models = []
for model_i in range(0, num_models):
    all_pair_embeddings, all_labels = build_train_test_df(idx_to_embedding, idx_to_label, num_pairs)

    # Training Loop
    dataset = PointerPairedDataset(idx_to_embedding, all_pair_embeddings, all_labels)
    dataloader = DataLoader(dataset, batch_size=batch_size, num_workers=20)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cl_model = SimpleNetwork(input_dim, hidden_dim, latent_dim).to(device)
    criterion =  ContrastiveLoss() #nn.CosineEmbeddingLoss() could just use cosine or could make your own
    optimizer = torch.optim.Adam(cl_model.parameters(), lr=lr)
    all_labels = np.array(all_labels)
    num_pos = (all_labels == 1).sum().item()
    num_neg = (all_labels == -1).sum().item()
    total = num_pos + num_neg
    pos_weight = total / (num_pos)
    neg_weight = total / (num_neg)#-> make it more balanced

    for epoch in range(epochs):
        for x1, x2, label in tqdm(dataloader, desc=f"Epoch {epoch + 1}"):
            x1, x2, label = x1.to(device), x2.to(device), label.to(device)
            out1 = cl_model(x1)
            out2 = cl_model(x2)
            # multiply by some value to make easier to see
            loss = 10000 * criterion(out1.to('cuda').squeeze(), out2.to('cuda').squeeze(), label.to('cuda'), 0.5, pos_weight, neg_weight)
            optimizer.zero_grad() #criterion is the contrastive loss close gets closer and the other further away
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch + 1}, Loss: {loss.item():.4f}")

    models.append(cl_model)

100%|██████████| 1000000/1000000 [00:03<00:00, 262655.98it/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 20 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Epoch 1:   0%|          | 0/1000 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 20 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if neces

Epoch 1, Loss: 183.6335


Epoch 2: 100%|██████████| 1000/1000 [00:35<00:00, 27.94it/s]


Epoch 2, Loss: 159.3009


Epoch 3: 100%|██████████| 1000/1000 [00:36<00:00, 27.62it/s]


Epoch 3, Loss: 153.6545


Epoch 4: 100%|██████████| 1000/1000 [00:34<00:00, 29.17it/s]


Epoch 4, Loss: 255.4334


Epoch 5: 100%|██████████| 1000/1000 [00:35<00:00, 27.84it/s]

Epoch 5, Loss: 117.2827


# Evaluate the held out test set

You should be able to get something akin to:

```
EC 1: 0.88
EC 2: 0.79
EC 3: 0.74
EC 4: 0.60
```
Note this is very similar to the SOTA model CLEAN!

# Test and evaluate on the test dataset

In [ ]:
cuda = True
DEVICE = torch.device("cuda" if cuda else "cpu")

data, labels = [], []
for i, (entry, ec) in enumerate(train_df[['Entry', 'EC number']].values):
    if seq_to_embedding.get(entry) is not None:
        clipped_embedding = seq_to_embedding.get(entry).flatten()
        data.append(clipped_embedding)
        ec = np.array([int(e.replace('n', '')) for e in ec.split('.')])
        labels.append(ec)

training_data = torch.tensor(np.array(data)).to(DEVICE)
training_labels = torch.tensor(np.array(labels)).to(DEVICE)
train_dataset = BasicDataset(training_data, training_labels)

data, labels = [], []
for i, (entry, ec) in enumerate(test_df[['Entry', 'EC number']].values):
    if seq_to_embedding.get(entry) is not None:
        clipped_embedding = seq_to_embedding.get(entry).flatten()
        data.append(clipped_embedding)
        ec = np.array([int(e.replace('n', '')) for e in ec.split('.')])
        labels.append(ec)

test_data = torch.tensor(np.array(data)).to(DEVICE)
test_labels = torch.tensor(np.array(labels)).to(DEVICE)
test_dataset = BasicDataset(test_data, test_labels)

# Have a train and test loader (note just have a single batch size for the test and don't shuffle)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [ ]:
for model in models:
    print('--------------- Evaluating model ---------------------')
    with torch.no_grad():
        train_embs, train_labels = compute_all_train_embeddings(model, train_dataset)

        correct_level4, correct_level3, correct_level2, correct_level1 = 0, 0, 0, 0
        total = 0
        train_embs_np = torch.stack(train_embs).cpu().numpy().squeeze(1) # Need to squeeze otherwise have an extra dim

        train_labels_np = train_labels

        for x, y_true in tqdm(test_dataset):
            test_emb = model.forward(x.unsqueeze(0)).squeeze(0).cpu().numpy()
            # Calculate the distance in the embedding space and why
            distances = np.linalg.norm(train_embs_np - test_emb, axis=1)
            topk_idx = np.argmin(distances)
            y_pred = [str(int(x)) for x in list(train_labels_np[topk_idx].cpu().numpy())]
            y_true = [str(int(x)) for x in list(y_true.cpu().numpy())]
            # Compute the accuracy for each level
            correct_level1 += 1 if y_pred[0] == y_true[0] else 0
            correct_level2 += 1 if ''.join(y_pred[:2]) == ''.join(y_true[:2]) else 0
            correct_level3 += 1 if ''.join(y_pred[:3]) == ''.join(y_true[:3]) else 0
            correct_level4 += 1 if ''.join(y_pred) == ''.join(y_true) else 0
            total += 1



--------------- Evaluating model ---------------------


100%|██████████| 432/432 [01:30<00:00,  4.78it/s]


In [ ]:
    print('------------- Accuracy')
    print('EC1 k=1 acc: ', correct_level1/total)
    print('EC2 k=1 acc: ', correct_level2/total)
    print('EC3 k=1 acc: ', correct_level3/total)
    print('EC4 k=1 acc: ', correct_level4/total)
    print('-------------')


------------- Accuracy
EC1 k=1 acc:  0.7476851851851852
EC2 k=1 acc:  0.6458333333333334
EC3 k=1 acc:  0.5856481481481481
EC4 k=1 acc:  0.44212962962962965
-------------
